# RNAAPIA — Face Recognition Pipeline
**Estrutura do notebook:**
```
0. Setup
1. Análise do dataset — distribuição de tamanhos + mediana
2. MTCNN — alinhamento com tamanho baseado na mediana
3. Filtragem — remover identidades com poucas imagens + outliers
4. Dataset + DataLoaders
5. Fase A — CNN própria (do zero)
6. Fase B — ResNet-50 (do zero)
7. Comparação de resultados
8. Fine-tuning com dataset organizacional
9. Avaliação FAR / FRR / EER
10. Script de inferência local
```

## ⚠️ Antes de começar
1. Adiciona o dataset VGGFace2: **Add Data** → `hearfool/vggface2`
2. Ativa a GPU: **Settings → Accelerator → GPU T4**
3. Corre as células por ordem

---
## 0. Setup — GPU + Paths

In [ ]:
# Verifica se o ambiente tem acesso a GPU e mostra o hardware disponível.
# Esta confirmação é importante porque todo o pipeline de treino fica muito mais rápido em CUDA.
import torch, os

print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Sem GPU! Ativa em Settings → Accelerator → GPU')

In [ ]:
# Centraliza todos os caminhos usados ao longo do notebook.
# Assim fica mais simples reutilizar os dados alinhados, checkpoints e logs gerados nas etapas seguintes.
BASE = '/kaggle/working'

PATHS = {
    'raw'         : '/kaggle/input/datasets/hearfool/vggface2/train',
    'aligned'     : f'{BASE}/vggface2_aligned',
    'org_dataset' : f'{BASE}/org_dataset',
    'org_aligned' : f'{BASE}/org_aligned',
    'checkpoints' : f'{BASE}/checkpoints',
    'logs'        : f'{BASE}/logs',
}

for k, v in PATHS.items():
    if k != 'raw':
        os.makedirs(v, exist_ok=True)

# Verificar dataset
all_identities = sorted(os.listdir(PATHS['raw']))
print(f'✓ Paths configurados')
print(f'  Identidades disponíveis no VGGFace2: {len(all_identities)}')

In [ ]:
%%capture
# Instala as dependências necessárias para deteção facial, visualização e treino.
# O uso de %%capture evita poluir o notebook com mensagens longas de instalação.
!pip install facenet-pytorch==2.5.3
!pip install tqdm matplotlib scikit-learn
print('✓ Dependências instaladas')

---
## 1. Análise do Dataset — Distribuição de Tamanhos
Antes de alinhar as faces, analisamos os tamanhos originais das imagens do VGGFace2.
O objetivo é encontrar a **mediana** e usá-la como target para o MTCNN,
evitando o downscaling excessivo que ocorre com o tamanho padrão de 112×112.

In [ ]:
# Recolhe uma amostra de imagens para perceber a distribuição real de larguras e alturas do dataset.
# Esta análise serve de base para escolher um IMAGE_SIZE mais ajustado antes do alinhamento facial.
import glob
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

# Amostra de N identidades para análise rápida
N_SAMPLE_IDS  = 50
N_SAMPLE_IMGS = 20

widths, heights = [], []

for identity in tqdm(all_identities[:N_SAMPLE_IDS], desc='A analisar tamanhos'):
    imgs = (glob.glob(os.path.join(PATHS['raw'], identity, '*.jpg')) +
            glob.glob(os.path.join(PATHS['raw'], identity, '*.png')))[:N_SAMPLE_IMGS]
    for p in imgs:
        try:
            w, h = Image.open(p).size
            widths.append(w)
            heights.append(h)
        except Exception:
            pass

widths  = np.array(widths)
heights = np.array(heights)

print(f'Imagens analisadas: {len(widths)}')
print(f'Largura  — min: {widths.min()} | mediana: {int(np.median(widths))} | max: {widths.max()} | média: {widths.mean():.0f}')
print(f'Altura   — min: {heights.min()} | mediana: {int(np.median(heights))} | max: {heights.max()} | média: {heights.mean():.0f}')

In [ ]:
# Desenha histogramas para comparar a distribuição de largura e altura das imagens originais.
# A mediana e a média ajudam a perceber qual o tamanho mais representativo do conjunto de dados.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, vals, label, color in [
    (axes[0], widths,  'Largura (px)', '#0077b6'),
    (axes[1], heights, 'Altura (px)',  '#2e4057'),
]:
    ax.hist(vals, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(np.median(vals), color='red',    linestyle='--', linewidth=2, label=f'Mediana: {int(np.median(vals))}px')
    ax.axvline(np.mean(vals),   color='orange', linestyle='--', linewidth=2, label=f'Média: {vals.mean():.0f}px')
    ax.set_xlabel(label); ax.set_ylabel('Frequência')
    ax.set_title(f'Distribuição — {label}')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('VGGFace2 — Tamanhos Originais das Imagens', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Define o tamanho alvo do pipeline a partir da mediana observada.
# O arredondamento para múltiplos de 32 mantém compatibilidade conveniente com arquiteturas convolucionais.
# Definir tamanho alvo: mediana arredondada para múltiplo de 32
raw_target = min(int(np.median(widths)), int(np.median(heights)))
IMAGE_SIZE = max(32, (raw_target // 32) * 32)

print(f'Mediana largura:  {int(np.median(widths))}px')
print(f'Mediana altura:   {int(np.median(heights))}px')
print(f'\n✓ IMAGE_SIZE definido: {IMAGE_SIZE}×{IMAGE_SIZE}px')

---
## 2. MTCNN — Alinhamento com Tamanho Baseado na Mediana
Retoma automaticamente se a sessão cair — identidades já processadas são saltadas.

In [ ]:
# Configura o MTCNN para detetar, cortar e alinhar rostos com o tamanho calculado anteriormente.
# A margem adicional preserva contexto útil em torno da face sem crescer demasiado a imagem final.
from facenet_pytorch import MTCNN
import shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
margin = max(20, IMAGE_SIZE // 6)

mtcnn = MTCNN(
    image_size=IMAGE_SIZE, margin=margin,
    min_face_size=20, thresholds=[0.6, 0.7, 0.7],
    factor=0.709, post_process=False, device=device
)
print(f'✓ MTCNN — image_size: {IMAGE_SIZE}px | margin: {margin}px | device: {device}')

In [ ]:
# Seleciona quantas identidades entram nesta experiência e descobre o que ainda falta processar.
# Isto permite retomar execuções interrompidas sem voltar a alinhar pastas já concluídas.
N_IDENTITIES            = 120
MIN_IMAGES_PER_IDENTITY = 20

identities   = all_identities[:N_IDENTITIES]
already_done = set(os.listdir(PATHS['aligned']))
remaining    = [i for i in identities if i not in already_done]

print(f'Total: {len(identities)} | Já feitas: {len(already_done)} | Por fazer: {len(remaining)}')

In [ ]:
# Percorre cada identidade pendente e guarda apenas as imagens em que o MTCNN encontrou uma face válida.
# O bloco também contabiliza falhas para dar feedback da qualidade do alinhamento.
failed, total_saved = 0, 0

for identity in tqdm(remaining, desc='MTCNN'):
    out_dir = os.path.join(PATHS['aligned'], identity)
    os.makedirs(out_dir, exist_ok=True)
    img_paths = (glob.glob(os.path.join(PATHS['raw'], identity, '*.jpg')) +
                 glob.glob(os.path.join(PATHS['raw'], identity, '*.png')))
    saved = 0
    for img_path in img_paths:
        try:
            face = mtcnn(Image.open(img_path).convert('RGB'))
            if face is not None:
                Image.fromarray(face.permute(1,2,0).byte().numpy()).save(
                    os.path.join(out_dir, os.path.basename(img_path)))
                saved += 1
        except Exception:
            failed += 1
    if saved < MIN_IMAGES_PER_IDENTITY:
        shutil.rmtree(out_dir)
    else:
        total_saved += saved

print(f'\n✓ MTCNN concluído!')
print(f'  Identidades: {len(os.listdir(PATHS["aligned"]))} | Imagens: {total_saved} | Falhas: {failed}')

---
## 3. Filtragem e Remoção de Outliers
Remove identidades com demasiadas ou poucas imagens (método IQR) e equilibra o dataset.

In [ ]:
# Mede quantas imagens existem por identidade depois do alinhamento.
# Esta distribuição ajuda a perceber se o dataset ficou desequilibrado entre pessoas.
# Análise da distribuição por identidade
id_counts = {d: len(os.listdir(os.path.join(PATHS['aligned'], d)))
             for d in os.listdir(PATHS['aligned'])}
counts = np.array(list(id_counts.values()))

print(f'Identidades: {len(counts)} | Imagens total: {counts.sum()}')
print(f'Por identidade — min: {counts.min()} | mediana: {int(np.median(counts))} | max: {counts.max()} | média: {counts.mean():.1f}')

plt.figure(figsize=(10, 4))
plt.hist(counts, bins=30, color='#0077b6', alpha=0.8, edgecolor='white')
plt.axvline(np.median(counts), color='red',    linestyle='--', linewidth=2, label=f'Mediana: {int(np.median(counts))}')
plt.axvline(np.mean(counts),   color='orange', linestyle='--', linewidth=2, label=f'Média: {counts.mean():.0f}')
plt.xlabel('Nº imagens por identidade'); plt.ylabel('Frequência')
plt.title('Distribuição pós-MTCNN'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig(f'{BASE}/identity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Remove identidades com contagens demasiado baixas ou demasiado altas usando o critério IQR.
# O objetivo é reduzir ruído e evitar que classes muito desbalanceadas dominem o treino.
# Remover outliers pelo método IQR
Q1, Q3 = np.percentile(counts, 25), np.percentile(counts, 75)
IQR    = Q3 - Q1
lower  = max(MIN_IMAGES_PER_IDENTITY, Q1 - 1.5 * IQR)
upper  = Q3 + 1.5 * IQR

print(f'Limites IQR — inferior: {lower:.0f} | superior: {upper:.0f}')

removed = 0
for identity, n in id_counts.items():
    if n < lower or n > upper:
        shutil.rmtree(os.path.join(PATHS['aligned'], identity))
        removed += 1

print(f'Outliers removidos: {removed}')

# Equilibrar: limitar ao máximo da mediana por identidade
import random
random.seed(42)
MAX_PER_ID = int(np.median([len(os.listdir(os.path.join(PATHS['aligned'], d)))
                            for d in os.listdir(PATHS['aligned'])]))

for identity in os.listdir(PATHS['aligned']):
    id_path = os.path.join(PATHS['aligned'], identity)
    imgs    = os.listdir(id_path)
    if len(imgs) > MAX_PER_ID:
        for f in random.sample(imgs, len(imgs) - MAX_PER_ID):
            os.remove(os.path.join(id_path, f))

final_ids  = os.listdir(PATHS['aligned'])
final_imgs = sum(len(os.listdir(os.path.join(PATHS['aligned'], d))) for d in final_ids)
print(f'\n✓ Dataset final: {len(final_ids)} identidades | {final_imgs} imagens | máx {MAX_PER_ID}/identidade')

---
## 4. Dataset + DataLoaders

In [ ]:
# Define transformações de treino, validação e teste e cria os respetivos DataLoaders.
# O augmentation aumenta a robustez do modelo a variações visuais frequentes em rostos.
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.1),
    transforms.RandomApply([transforms.RandomRotation(15)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_dataset = datasets.ImageFolder(PATHS['aligned'], transform=train_transform)
NUM_CLASSES  = len(full_dataset.classes)
total        = len(full_dataset)
n_train      = int(0.70 * total)
n_val        = int(0.15 * total)
n_test       = total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'IMAGE_SIZE: {IMAGE_SIZE}×{IMAGE_SIZE}px | Classes: {NUM_CLASSES}')
print(f'Total: {total} | Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Batches por época: {len(train_loader)}')

---
## 5. Fase A — FaceCNN Própria (do Zero)
Arquitetura desenhada de raiz: 4 blocos Conv→BN→ReLU→MaxPool + embedding FC(512) + classificador FC(N).

In [ ]:
# Implementa a CNN própria usada como baseline do projeto.
# A arquitetura aprende embeddings faciais e termina numa camada classificadora para treino supervisionado.
import torch.nn as nn
import torch.nn.functional as F
import json, time
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

class ConvBlock(nn.Module):
    """Bloco convolucional: Conv2d → BatchNorm → ReLU → MaxPool"""
    def __init__(self, in_ch, out_ch, kernel=3, pool=2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=kernel//2, bias=False),  # Extrai padrões locais como contornos e texturas faciais.
            nn.BatchNorm2d(out_ch),  # Estabiliza a distribuição das ativações entre batches.
            nn.ReLU(inplace=True),  # Introduz não linearidade para a rede aprender padrões complexos.
            nn.MaxPool2d(pool)  # Reduz a resolução espacial e mantém as respostas mais fortes.
        )
    def forward(self, x): return self.block(x)


# FaceCNN: 4 camadas convolucionais (nos 4 ConvBlock) + 2 camadas densas (embedding + classifier)
class FaceCNN(nn.Module):
    """
    CNN própria desenhada de raiz para reconhecimento facial.
    Entrada: IMAGE_SIZE × IMAGE_SIZE × 3
    Saída (treino):     logits N_classes
    Saída (inferência): embedding 512-d normalizado
    """
    def __init__(self, num_classes, embedding_size=512, image_size=224):
        super().__init__()
        # Parte convolucional da FaceCNN: 4 camadas Conv2d no total.
        self.conv_blocks = nn.Sequential(
            ConvBlock(3,   32),   # → size/2
            ConvBlock(32,  64),   # → size/4
            ConvBlock(64,  128),  # → size/8
            ConvBlock(128, 256),  # → size/16
        )
        fm_size   = image_size // 16  # Quatro blocos com MaxPool(2) reduzem a dimensão espacial por 2^4.
        flat_size = 256 * fm_size * fm_size  # Número total de features antes da parte totalmente ligada.
        # Parte densa da FaceCNN: 1 camada Linear para embedding.
        # Primeira camada densa adicional da ResNet50Face: projeta as features para o embedding facial.
        self.embedding = nn.Sequential(
            nn.Flatten(),  # Converte o mapa de features 3D num vetor 1D por imagem.
            nn.Dropout(p=0.4),  # Desliga neurónios aleatoriamente para reduzir overfitting.
            nn.Linear(flat_size, embedding_size),  # Projeta as features convolucionais num espaço compacto de identidade.
            nn.BatchNorm1d(embedding_size),  # Normaliza o embedding antes da ativação final.
            nn.ReLU(inplace=True)  # Garante não linearidade na representação aprendida.
        )
        # Segunda camada densa da FaceCNN: classificação final por identidade.
        # Segunda camada densa adicional da ResNet50Face: classificação final.
        self.classifier = nn.Linear(embedding_size, num_classes)  # Converte o embedding em logits, um score por identidade.

    def forward(self, x, return_embedding=False):
        x   = self.conv_blocks(x)  # Extrai mapas de características hierárquicos da face.
        emb = self.embedding(x)  # Resume esses mapas num vetor compacto de identidade.
        if return_embedding:
            return F.normalize(emb, dim=1)  # Normalização L2 útil para comparar embeddings com similaridade do cosseno.
        return self.classifier(emb)  # Durante treino supervisionado devolvemos logits para a loss de classificação.


EMBEDDING_SIZE = 512
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cnn_model = FaceCNN(NUM_CLASSES, EMBEDDING_SIZE, IMAGE_SIZE).to(device)
print(f'✓ FaceCNN — {sum(p.numel() for p in cnn_model.parameters())/1e6:.2f}M params')

In [ ]:
# Reúne funções genéricas de treino e avaliação para reutilizar nos dois modelos.
# Isto evita duplicação de lógica e garante uma comparação mais justa entre arquiteturas.
# Funções de treino e avaliação (partilhadas entre CNN e ResNet)

def train_one_epoch(model, loader, optimizer, scaler, device):
    model.train()  # Ativa dropout e batch norm em modo de treino.
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)  # Move batch e rótulos para GPU/CPU.
        optimizer.zero_grad()  # Limpa gradientes acumulados do batch anterior.
        with torch.amp.autocast('cuda'):
            out  = model(imgs)  # Forward pass: gera logits para cada identidade.
            loss = F.cross_entropy(out, labels)  # Mede o erro entre logits previstos e classe correta.
        scaler.scale(loss).backward()  # Backpropagation com AMP para evitar underflow em float16.
        scaler.step(optimizer)  # Atualiza os pesos com os gradientes calculados.
        scaler.update()  # Ajusta dinamicamente o fator de escala do GradScaler.
        total_loss += loss.item() * imgs.size(0)  # Soma ponderada para média final por imagem.
        correct    += (out.argmax(1) == labels).sum().item()  # Conta previsões corretas no batch.
        total      += imgs.size(0)  # Conta quantas imagens já foram processadas.
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()  # Desativa dropout e usa estatísticas acumuladas do batch norm.
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Val', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs).float()  # Forward em validação; convertemos para float32 para cálculo estável da loss.
        loss = F.cross_entropy(out, labels)
        if not torch.isnan(loss):
            total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return total_loss / total, correct / total

def save_checkpoint(path, epoch, model, optimizer, scheduler, history):
    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'history': history}, path)

print('✓ Funções prontas')

In [ ]:
# Inicializa o ciclo de treino da FaceCNN, incluindo otimizador, scheduler, AMP e retoma de checkpoint.
# O histórico guardado em disco permite continuar experiências longas sem perder progresso.
CNN_EPOCHS    = 30
cnn_optimizer = Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)  # Adam combina momentum e adaptação individual do learning rate.
cnn_scheduler = CosineAnnealingLR(cnn_optimizer, T_max=CNN_EPOCHS, eta_min=1e-6)  # Faz o learning rate descer suavemente ao longo das épocas.
cnn_scaler    = torch.amp.GradScaler('cuda')  # Permite mixed precision com mais estabilidade numérica.

CNN_CKPT  = f"{PATHS['checkpoints']}/cnn_latest.pth"
START_CNN = 0
cnn_hist  = []

if os.path.exists(CNN_CKPT):
    ckpt = torch.load(CNN_CKPT, map_location=device)
    cnn_model.load_state_dict(ckpt['model'])
    cnn_optimizer.load_state_dict(ckpt['optimizer'])
    cnn_scheduler.load_state_dict(ckpt['scheduler'])
    START_CNN = ckpt['epoch'] + 1
    cnn_hist  = ckpt.get('history', [])
    print(f'✓ A retomar CNN do epoch {START_CNN}')
else:
    print('A iniciar treino CNN do zero.')

best_cnn_acc = max((h['val_acc'] for h in cnn_hist), default=0.0)

for epoch in range(START_CNN, CNN_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(cnn_model, train_loader, cnn_optimizer, cnn_scaler, device)  # Aprende com batches augmentados.
    val_loss,   val_acc   = evaluate(cnn_model, val_loader, device)  # Mede generalização em dados não vistos no treino.
    cnn_scheduler.step()  # Atualiza o learning rate para a época seguinte.
    elapsed = time.time() - t0

    record = {'epoch': epoch, 'train_loss': round(train_loss,4), 'train_acc': round(train_acc,4),
              'val_loss': round(val_loss,4), 'val_acc': round(val_acc,4), 'time_s': round(elapsed,1)}
    cnn_hist.append(record)

    print(f'[CNN] {epoch+1:02d}/{CNN_EPOCHS} | '
          f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
          f'Val: {val_loss:.4f} ({val_acc*100:.1f}%) | {elapsed:.0f}s')

    save_checkpoint(CNN_CKPT, epoch, cnn_model, cnn_optimizer, cnn_scheduler, cnn_hist)

    if val_acc > best_cnn_acc:
        best_cnn_acc = val_acc
        save_checkpoint(f'{BASE}/best_cnn.pth', epoch, cnn_model, cnn_optimizer, cnn_scheduler, cnn_hist)
        print(f'  ✓ Melhor CNN guardada ({val_acc*100:.2f}%)')

    with open(f"{PATHS['logs']}/cnn_log.json", 'w') as f:
        json.dump(cnn_hist, f, indent=2)

print(f'\n✓ FaceCNN concluída! Melhor val_acc: {best_cnn_acc*100:.2f}%')

---
## 6. Fase B — ResNet-50 (do Zero)
Mesmos dados, mesmas condições — para comparação direta com a FaceCNN.

In [ ]:
# Define a variante ResNet-50 sem pesos pré-treinados para comparar com a CNN própria.
# O backbone extrai características profundas e a cabeça final adapta-se ao número de identidades.
import torchvision.models as tv_models

# ResNet50Face: backbone ResNet-50 com 49 camadas convolucionais + 2 camadas densas no topo (embedding + classifier)
class ResNet50Face(nn.Module):
    """ResNet-50 sem pesos pré-treinados, com camada de embedding 512-d."""
    def __init__(self, num_classes, embedding_size=512):
        super().__init__()
        # Parte convolucional da ResNet50Face: as convoluções ficam dentro do backbone ResNet-50.
        backbone       = tv_models.resnet50(weights=None)  # Cria a ResNet-50 sem pré-treino para comparação justa.
        self.features  = nn.Sequential(*list(backbone.children())[:-1])  # Remove a camada final original e fica só o extrator de features.
        # Primeira camada densa adicional da ResNet50Face: projeta as features para o embedding facial.
        self.embedding = nn.Sequential(
            nn.Flatten(),  # Achata a saída global do backbone.
            nn.Dropout(p=0.4),  # Regulariza a cabeça final.
            nn.Linear(2048, embedding_size),  # Comprime as 2048 features da ResNet para o embedding facial.
            nn.BatchNorm1d(embedding_size),  # Estabiliza o espaço de embeddings.
            nn.ReLU(inplace=True)
        )
        # Segunda camada densa adicional da ResNet50Face: classificação final.
        self.classifier = nn.Linear(embedding_size, num_classes)

    def forward(self, x, return_embedding=False):
        x   = self.features(x)  # O backbone residual extrai características profundas da face.
        emb = self.embedding(x)  # A cabeça adapta essas features ao espaço de embedding do projeto.
        if return_embedding:
            return F.normalize(emb, dim=1)  # Embeddings normalizados funcionam melhor com similaridade do cosseno.
        return self.classifier(emb)


resnet_model = ResNet50Face(NUM_CLASSES, EMBEDDING_SIZE).to(device)
print(f'✓ ResNet50Face — {sum(p.numel() for p in resnet_model.parameters())/1e6:.1f}M params')

In [ ]:
# Configura o treino da ResNet-50 nas mesmas condições da FaceCNN.
# Manter o protocolo semelhante ajuda a comparar os resultados de forma consistente.
RN_EPOCHS    = 30
rn_optimizer = Adam(resnet_model.parameters(), lr=1e-3, weight_decay=1e-4)  # Mesmo otimizador da CNN para comparação justa.
rn_scheduler = CosineAnnealingLR(rn_optimizer, T_max=RN_EPOCHS, eta_min=1e-6)  # Faz annealing do learning rate ao longo do treino.
rn_scaler    = torch.amp.GradScaler('cuda')  # Mixed precision acelera o treino em GPU.

RN_CKPT  = f"{PATHS['checkpoints']}/resnet_latest.pth"
START_RN = 0
rn_hist  = []

if os.path.exists(RN_CKPT):
    ckpt = torch.load(RN_CKPT, map_location=device)
    resnet_model.load_state_dict(ckpt['model'])
    rn_optimizer.load_state_dict(ckpt['optimizer'])
    rn_scheduler.load_state_dict(ckpt['scheduler'])
    START_RN = ckpt['epoch'] + 1
    rn_hist  = ckpt.get('history', [])
    print(f'✓ A retomar ResNet do epoch {START_RN}')
else:
    print('A iniciar treino ResNet do zero.')

best_rn_acc = max((h['val_acc'] for h in rn_hist), default=0.0)

for epoch in range(START_RN, RN_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(resnet_model, train_loader, rn_optimizer, rn_scaler, device)
    val_loss,   val_acc   = evaluate(resnet_model, val_loader, device)
    rn_scheduler.step()
    elapsed = time.time() - t0

    record = {'epoch': epoch, 'train_loss': round(train_loss,4), 'train_acc': round(train_acc,4),
              'val_loss': round(val_loss,4), 'val_acc': round(val_acc,4), 'time_s': round(elapsed,1)}
    rn_hist.append(record)

    print(f'[ResNet] {epoch+1:02d}/{RN_EPOCHS} | '
          f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
          f'Val: {val_loss:.4f} ({val_acc*100:.1f}%) | {elapsed:.0f}s')

    save_checkpoint(RN_CKPT, epoch, resnet_model, rn_optimizer, rn_scheduler, rn_hist)

    if val_acc > best_rn_acc:
        best_rn_acc = val_acc
        save_checkpoint(f'{BASE}/best_resnet.pth', epoch, resnet_model, rn_optimizer, rn_scheduler, rn_hist)
        print(f'  ✓ Melhor ResNet guardada ({val_acc*100:.2f}%)')

    with open(f"{PATHS['logs']}/resnet_log.json", 'w') as f:
        json.dump(rn_hist, f, indent=2)

print(f'\n✓ ResNet concluída! Melhor val_acc: {best_rn_acc*100:.2f}%')

---
## 7. Comparação de Resultados — FaceCNN vs ResNet-50

In [ ]:
# Carrega os logs dos dois treinos e plota a evolução das métricas principais.
# Esta comparação visual facilita perceber qual modelo generaliza melhor.
with open(f"{PATHS['logs']}/cnn_log.json")    as f: cnn_hist = json.load(f)
with open(f"{PATHS['logs']}/resnet_log.json") as f: rn_hist  = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, key, ylabel, title in [
    (axes[0], 'val_loss', 'Cross-Entropy Loss', 'Val Loss'),
    (axes[1], 'val_acc',  'Accuracy (%)',        'Val Accuracy'),
]:
    scale = 100 if key == 'val_acc' else 1
    ax.plot([h['epoch']+1 for h in cnn_hist], [h[key]*scale for h in cnn_hist],
            'b-o', markersize=3, label='FaceCNN (própria)')
    ax.plot([h['epoch']+1 for h in rn_hist],  [h[key]*scale for h in rn_hist],
            'r-o', markersize=3, label='ResNet-50')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(f'{title} — FaceCNN vs ResNet-50')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Comparação — Mesmo Dataset, Treino do Zero', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_cnn = max(cnn_hist, key=lambda h: h['val_acc'])
best_rn  = max(rn_hist,  key=lambda h: h['val_acc'])
cnn_p    = sum(p.numel() for p in cnn_model.parameters()) / 1e6
rn_p     = sum(p.numel() for p in resnet_model.parameters()) / 1e6

print('\n── Resumo ───────────────────────────────────────────')
print(f'{"":22s} {"FaceCNN":>12s} {"ResNet-50":>12s}')
print(f'{"Parâmetros":22s} {cnn_p:>11.2f}M {rn_p:>11.1f}M')
print(f'{"Melhor val_acc":22s} {best_cnn["val_acc"]*100:>11.2f}% {best_rn["val_acc"]*100:>11.2f}%')
print(f'{"Epoch (melhor)":22s} {best_cnn["epoch"]+1:>12d} {best_rn["epoch"]+1:>12d}')
print(f'{"Tempo/época (s)":22s} {best_cnn["time_s"]:>12.0f} {best_rn["time_s"]:>12.0f}')

---
## 8. Fine-Tuning com Dataset Organizacional
**Antes de correr esta secção**, coloca as fotos em `/kaggle/working/org_dataset/`:
```
org_dataset/
    pessoa1/   ← 5–10 fotos .jpg
    pessoa2/
    ...
```
O fine-tuning usa o melhor modelo da comparação anterior como ponto de partida.

In [ ]:
# Valida se o dataset organizacional foi colocado na pasta esperada e resume o número de imagens por pessoa.
# Esta verificação evita avançar para o fine-tuning com dados em falta ou estrutura incorreta.
# Verificar fotos da organização
people = [d for d in os.listdir(PATHS['org_dataset'])
          if os.path.isdir(os.path.join(PATHS['org_dataset'], d))]

if not people:
    print(f'⚠️  Sem pessoas em {PATHS["org_dataset"]}')
else:
    print(f'Pessoas: {people}')
    for p in people:
        imgs = (glob.glob(os.path.join(PATHS['org_dataset'], p, '*.jpg')) +
                glob.glob(os.path.join(PATHS['org_dataset'], p, '*.png')))
        print(f'  {p}: {len(imgs)} imagens')

In [ ]:
# Alinha as fotografias da organização com o mesmo MTCNN usado no dataset principal.
# Reutilizar o mesmo pré-processamento mantém coerência entre treino base e fine-tuning.
# Alinhar faces org com MTCNN
for person in tqdm(people, desc='MTCNN org'):
    out_dir = os.path.join(PATHS['org_aligned'], person)
    os.makedirs(out_dir, exist_ok=True)
    if len(os.listdir(out_dir)) > 5:
        print(f'  {person}: já processado'); continue
    img_paths = (glob.glob(os.path.join(PATHS['org_dataset'], person, '*.jpg')) +
                 glob.glob(os.path.join(PATHS['org_dataset'], person, '*.png')))
    saved = 0
    for img_path in img_paths:
        try:
            face = mtcnn(Image.open(img_path).convert('RGB'))
            if face is not None:
                Image.fromarray(face.permute(1,2,0).byte().numpy()).save(
                    os.path.join(out_dir, os.path.basename(img_path)))
                saved += 1
        except Exception: pass
    print(f'  {person}: {saved} faces alinhadas')

In [ ]:
# Cria variações artificiais das imagens organizacionais para compensar o número reduzido de exemplos reais.
# Este aumento de dados ajuda o modelo a adaptar-se melhor às novas identidades.
# Data augmentation — gera ~20 variações por imagem original
import random

aug_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.3),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
])

AUG_DIR    = f'{BASE}/org_augmented'
N_AUGMENTS = 20
os.makedirs(AUG_DIR, exist_ok=True)

for person in people:
    out_dir = os.path.join(AUG_DIR, person)
    os.makedirs(out_dir, exist_ok=True)
    aligned = (glob.glob(os.path.join(PATHS['org_aligned'], person, '*.jpg')) +
               glob.glob(os.path.join(PATHS['org_aligned'], person, '*.png')))
    for i, img_path in enumerate(aligned):
        img = Image.open(img_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        img.save(os.path.join(out_dir, f'orig_{i:03d}.jpg'))
        for j in range(N_AUGMENTS):
            aug_transform(img).save(os.path.join(out_dir, f'aug_{i:03d}_{j:02d}.jpg'))
    print(f'  {person}: {len(aligned)} originais → {len(os.listdir(out_dir))} total')

In [ ]:
# Monta os DataLoaders específicos do fine-tuning sobre o dataset organizacional aumentado.
# A divisão treino/validação/teste permite medir adaptação sem perder capacidade de avaliação.
# DataLoaders para fine-tuning
ft_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

org_full        = datasets.ImageFolder(AUG_DIR, transform=ft_transform)
ORG_NUM_CLASSES = len(org_full.classes)
n_ft_train      = int(0.70 * len(org_full))
n_ft_val        = int(0.15 * len(org_full))
n_ft_test       = len(org_full) - n_ft_train - n_ft_val

ft_train, ft_val, ft_test = random_split(
    org_full, [n_ft_train, n_ft_val, n_ft_test],
    generator=torch.Generator().manual_seed(42))

ft_train_loader = DataLoader(ft_train, batch_size=16, shuffle=True,  num_workers=2)
ft_val_loader   = DataLoader(ft_val,   batch_size=16, shuffle=False, num_workers=2)
ft_test_loader  = DataLoader(ft_test,  batch_size=16, shuffle=False, num_workers=2)

print(f'Pessoas: {org_full.classes} | Total: {len(org_full)}')
print(f'Train: {n_ft_train} | Val: {n_ft_val} | Test: {n_ft_test}')

In [ ]:
# Escolhe o modelo de partida para o fine-tuning e carrega os melhores pesos da fase anterior.
# Depois substitui a camada classificadora para refletir apenas as pessoas da organização.
# Escolhe qual modelo usar para fine-tuning
USE_MODEL = 'cnn'  # 'cnn' ou 'resnet'

if USE_MODEL == 'cnn':
    ft_model = FaceCNN(NUM_CLASSES, EMBEDDING_SIZE, IMAGE_SIZE).to(device)
    ckpt     = torch.load(f'{BASE}/best_cnn.pth', map_location=device)
else:
    ft_model = ResNet50Face(NUM_CLASSES, EMBEDDING_SIZE).to(device)
    ckpt     = torch.load(f'{BASE}/best_resnet.pth', map_location=device)

ft_model.load_state_dict(ckpt['model'])  # Reaproveita pesos já treinados no dataset facial maior.

# Substituir classificador pelo número de pessoas da organização
ft_model.classifier = nn.Linear(EMBEDDING_SIZE, ORG_NUM_CLASSES).to(device)  # A última camada tem de refletir o novo número de classes.

# Congelar camadas convolucionais, treinar só embedding + classificador
for name, param in ft_model.named_parameters():
    param.requires_grad = any(l in name for l in ['embedding', 'classifier'])  # Mantém o extrator facial e ajusta apenas a parte mais específica.

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad) / 1e6
print(f'✓ Fine-tuning de {USE_MODEL.upper()} | Treináveis: {trainable:.2f}M | Classes org: {org_full.classes}')

In [ ]:
# Treina o modelo adaptado às identidades organizacionais e guarda o melhor checkpoint.
# O foco aqui é transfer learning: preservar conhecimento útil e ajustar apenas o necessário.
FT_EPOCHS    = 30
ft_optimizer = Adam([p for p in ft_model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)  # Otimiza apenas os pesos descongelados no fine-tuning.
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-6)  # Reduz gradualmente o learning rate na adaptação final.
ft_scaler    = torch.amp.GradScaler('cuda')
ft_hist      = []
best_ft_acc  = 0.0

for epoch in range(FT_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(ft_model, ft_train_loader, ft_optimizer, ft_scaler, device)
    val_loss,   val_acc   = evaluate(ft_model, ft_val_loader, device)
    ft_scheduler.step()
    elapsed = time.time() - t0
    ft_hist.append({'epoch': epoch, 'train_loss': round(train_loss,4),
                    'val_loss': round(val_loss,4), 'val_acc': round(val_acc,4)})
    print(f'[FT] {epoch+1:02d}/{FT_EPOCHS} | '
          f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
          f'Val: {val_loss:.4f} ({val_acc*100:.1f}%) | {elapsed:.0f}s')
    if val_acc > best_ft_acc:
        best_ft_acc = val_acc
        save_checkpoint(f'{BASE}/best_finetuned.pth', epoch, ft_model, ft_optimizer, ft_scheduler, ft_hist)
        print(f'  ✓ Melhor fine-tuned guardado ({val_acc*100:.2f}%)')

print(f'\n✓ Fine-tuning concluído! Melhor acc: {best_ft_acc*100:.2f}%')

---
## 9. Avaliação Final — FAR / FRR / EER

In [ ]:
# Calcula embeddings do conjunto de teste e avalia métricas biométricas como FAR, FRR e EER.
# Estas métricas são mais informativas do que accuracy simples em cenários de reconhecimento facial.
import numpy as np
from sklearn.metrics import classification_report

ckpt = torch.load(f'{BASE}/best_finetuned.pth', map_location=device)
ft_model.load_state_dict(ckpt['model'])
ft_model.eval()

# Embeddings do test set
all_emb, all_labels = [], []
with torch.no_grad():
    for imgs, labels in ft_test_loader:
        emb = ft_model(imgs.to(device), return_embedding=True)  # Extrai embeddings já normalizados para cada rosto do teste.
        all_emb.append(emb.cpu())
        all_labels.append(labels)
all_emb    = torch.cat(all_emb)
all_labels = torch.cat(all_labels)

# Gallery: embedding médio por pessoa
gallery = {}
with torch.no_grad():
    for imgs, labels in ft_train_loader:
        emb = ft_model(imgs.to(device), return_embedding=True).cpu()
        for e, l in zip(emb, labels):
            gallery.setdefault(l.item(), []).append(e)

gallery_means  = {k: F.normalize(torch.stack(v).mean(0).unsqueeze(0), dim=1).squeeze()  # Protótipo médio por identidade.
                  for k, v in gallery.items()}
gallery_matrix = torch.stack([gallery_means[i] for i in range(ORG_NUM_CLASSES)])  # Matriz com uma representação por pessoa.
similarities   = all_emb @ gallery_matrix.T  # Como tudo está normalizado, produto interno = cosine similarity.

max_scores  = similarities.max(dim=1).values.numpy()
pred_labels = similarities.argmax(dim=1).numpy()
true_labels = all_labels.numpy()

genuine_scores  = max_scores[pred_labels == true_labels]
impostor_scores = max_scores[pred_labels != true_labels]

thresholds = np.linspace(0, 1, 1000)
FARs = np.array([(impostor_scores >= t).mean() if len(impostor_scores) > 0 else 0.0 for t in thresholds])
FRRs = np.array([(genuine_scores  <  t).mean() if len(genuine_scores)  > 0 else 0.0 for t in thresholds])
eer_idx = np.argmin(np.abs(FARs - FRRs))
EER     = (FARs[eer_idx] + FRRs[eer_idx]) / 2
EER_t   = thresholds[eer_idx]

print(f'EER:           {EER*100:.2f}%')
print(f'Threshold EER: {EER_t:.3f}')
print(f'FAR @ EER:     {FARs[eer_idx]*100:.2f}%')
print(f'FRR @ EER:     {FRRs[eer_idx]*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thresholds, FARs*100, 'r-', label='FAR (%)', linewidth=2)
axes[0].plot(thresholds, FRRs*100, 'b-', label='FRR (%)', linewidth=2)
axes[0].axvline(EER_t, color='g', linestyle='--', label=f'EER={EER*100:.2f}% @ {EER_t:.2f}')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Rate (%)')
axes[0].set_title('FAR / FRR'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].hist(genuine_scores,  bins=30, alpha=0.6, color='green', label='Genuine',  density=True)
axes[1].hist(impostor_scores, bins=30, alpha=0.6, color='red',   label='Impostor', density=True)
axes[1].axvline(EER_t, color='black', linestyle='--', label='EER threshold')
axes[1].set_xlabel('Cosine Similarity'); axes[1].set_title('Score Distributions')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Avaliação FAR/FRR/EER — Dataset Organizacional', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/far_frr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

torch.save({org_full.classes[k]: v for k, v in gallery_means.items()}, f'{BASE}/gallery.pt')
print('\n✓ gallery.pt guardado')
mask = max_scores >= EER_t
if mask.sum() > 0:
    print('\nClassification Report @ EER threshold:')
    print(classification_report(true_labels[mask], pred_labels[mask],
                                target_names=org_full.classes, zero_division=0))

---
## 10. Script de Inferência Local

In [ ]:
# Gera um script autónomo de inferência em tempo real para reutilizar o modelo fora do notebook.
# Isto facilita transformar o resultado do treino numa demo prática de controlo de acesso.
script = f'''# realtime_access_control.py
# pip install torch torchvision facenet-pytorch opencv-python
# Ficheiros: best_finetuned.pth + gallery.pt

import torch, cv2
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from facenet_pytorch import MTCNN
from PIL import Image

CHECKPOINT = "best_finetuned.pth"
GALLERY    = "gallery.pt"
THRESHOLD  = {EER_t:.3f}
IMAGE_SIZE = {IMAGE_SIZE}

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2))
    def forward(self, x): return self.block(x)

# FaceCNN: 4 camadas convolucionais (nos 4 ConvBlock) + 2 camadas densas (embedding + classifier)
class FaceCNN(nn.Module):
    def __init__(self, num_classes, emb=512, sz=224):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            ConvBlock(3,32), ConvBlock(32,64), ConvBlock(64,128), ConvBlock(128,256))
        fm = sz // 16
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(256*fm*fm, emb), nn.BatchNorm1d(emb), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(emb, num_classes)
    def forward(self, x, return_embedding=False):
        x = self.conv_blocks(x)
        e = self.embedding(x)
        return F.normalize(e, dim=1) if return_embedding else self.classifier(e)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gallery = torch.load(GALLERY, map_location=device)
people  = list(gallery.keys())
gmat    = torch.stack([gallery[p] for p in people]).to(device)

model = FaceCNN(len(people), sz=IMAGE_SIZE).to(device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device)["model"])
model.eval()

mtcnn = MTCNN(image_size=IMAGE_SIZE, margin=IMAGE_SIZE//6, device=device)
tf    = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)])

cap = cv2.VideoCapture(0)
print(f"Sistema iniciado ({{device}}). Prima Q para sair.")

while True:
    ret, frame = cap.read()
    if not ret: break
    pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    boxes, probs = mtcnn.detect(pil)
    if boxes is not None:
        for box, prob in zip(boxes, probs):
            if prob is None or prob < 0.9: continue
            face = mtcnn(pil)
            if face is None: continue
            t = tf(Image.fromarray(face.permute(1,2,0).byte().numpy())).unsqueeze(0).to(device)
            with torch.no_grad():
                emb   = model(t, return_embedding=True)
                sims  = (emb @ gmat.T).squeeze()
                score = sims.max().item()
                idx   = sims.argmax().item()
            x1,y1,x2,y2 = [int(b) for b in box]
            if score >= THRESHOLD:
                color, label = (0,255,0), f"PERMITIDO: {{people[idx]}} ({{score:.2f}})"
            else:
                color, label = (0,0,255), f"NEGADO ({{score:.2f}})"
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,label,(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.65,color,2)
    cv2.imshow("Controlo de Acesso — RNAAPIA", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release()
cv2.destroyAllWindows()
'''

with open(f'{BASE}/realtime_access_control.py', 'w') as f:
    f.write(script)
print('✓ Script guardado')
print('\n1. Descarrega best_finetuned.pth e gallery.pt do Kaggle Output')
print('2. pip install torch torchvision facenet-pytorch opencv-python')
print('3. python realtime_access_control.py')

---
## 📁 Ficheiros gerados em `/kaggle/working/`
```
vggface2_aligned/          ← faces alinhadas com IMAGE_SIZE calculado
org_dataset/               ← fotos originais da organização
org_aligned/               ← faces org alinhadas
org_augmented/             ← faces org com augmentation (~20x)
checkpoints/
│   cnn_latest.pth         ← último checkpoint FaceCNN
│   resnet_latest.pth      ← último checkpoint ResNet-50
logs/
│   cnn_log.json
│   resnet_log.json
best_cnn.pth               ← melhor FaceCNN no VGGFace2
best_resnet.pth            ← melhor ResNet-50 no VGGFace2
best_finetuned.pth         ← modelo final (fine-tuned) ← DEMO
gallery.pt                 ← embeddings org            ← DEMO
size_distribution.png      ← análise tamanhos originais
identity_distribution.png  ← distribuição por identidade
comparison.png             ← FaceCNN vs ResNet-50
far_frr_curves.png
realtime_access_control.py
```

## 🔄 Se a sessão cair
Corre secções **0 → 4** e depois a secção onde ficaste —
os checkpoints carregam automaticamente.